**GOTabPFN in RELATHE dataset. Only 50 Optuna trials was used to get best hyperparameters for this dataset. The number of trials could help to search in more depth.**

In [ ]:
# =======================
# GPU selection (no masking)
# =======================
import os
GPU_ID = 3  # <-- run on cuda:3
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")

import gc
import time
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from gotabpfn import GraphFeatureOrdering, pidf_segpca, TabPFN25Head, TabPFN25Config
PIDFSegPCA = pidf_segpca

SEED = 42
DATA_FILE = "RELATHE.csv"
TARGET_COL = "label"

# ---- fixed best hyperparameters from Optuna trial 28 ----
BEST_PARAMS = {
    "go_metric": "manhattan",
    "go_num_clusters": 6,
    "go_refine_passes": 2,
    "go_direction_select": True,
    "go_feat_subsample": 2000,
    "nsc_segmentation": "uniform",
    "nsc_m_rule": "default",
    "nsc_tau": 0.95,
    "nsc_gamma": 2.116958048807466,
    "nsc_beta": 0.05755662243703091,
    "nsc_Mmin": 16,
    "nsc_Mmax": 384,
    "nsc_lmin": 12,
    "assume_standardized": True,
    "tabpfn_seed": 1,
}

# ---- set runtime device to cuda:3 (do NOT mask others) ----
if torch.cuda.is_available():
    torch.cuda.set_device(GPU_ID)
    DEVICE_STR = f"cuda:{GPU_ID}"
else:
    DEVICE_STR = "cpu"

NSC_DEVICE = DEVICE_STR
TABPFN_DEVICE = DEVICE_STR


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def cleanup_cuda():
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.synchronize()
        except Exception:
            pass
        torch.cuda.empty_cache()


def compute_deltas_adjacent_corr_chunked(
    X_tr: np.ndarray,
    Pi_star: list[int],
    chunk: int = 2048,
    eps: float = 1e-12,
):
    X = torch.from_numpy(X_tr).float()  # CPU
    perm = torch.tensor(Pi_star, dtype=torch.long)

    n, d = X.shape
    deltas = torch.empty(d - 1, dtype=torch.float32)

    mu = X.mean(dim=0)
    Xc = X - mu
    std = Xc.std(dim=0, unbiased=False).clamp_min(eps)

    for a in range(0, d - 1, chunk):
        b = min(d - 1, a + chunk)
        i_idx = perm[a:b]
        j_idx = perm[a + 1:b + 1]
        Zi = (Xc[:, i_idx] / std[i_idx]).to(torch.float32)
        Zj = (Xc[:, j_idx] / std[j_idx]).to(torch.float32)
        corr = (Zi * Zj).mean(dim=0)
        deltas[a:b] = (1.0 - corr.abs()).cpu()

    return deltas


def safe_binary_auc(y_true: np.ndarray, proba: np.ndarray):
    """
    For binary classification, use the positive-class probability if available.
    """
    try:
        if proba.ndim == 2 and proba.shape[1] >= 2:
            return float(roc_auc_score(y_true, proba[:, 1]))
        elif proba.ndim == 1:
            return float(roc_auc_score(y_true, proba))
        else:
            return np.nan
    except Exception:
        return np.nan


# ============================================
# Load + preprocessing
# Remove non-numeric categorical/object columns
# ============================================
seed_everything(SEED)
df = pd.read_csv(DATA_FILE)

if TARGET_COL not in df.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in {DATA_FILE}")

# ----- target label encoding -----
y_raw = df[TARGET_COL].astype(str).fillna("missing_target")
target_le = LabelEncoder()
y_all = target_le.fit_transform(y_raw).astype(np.int64)
class_map = {cls: int(i) for i, cls in enumerate(target_le.classes_)}
NUM_CLASSES = len(target_le.classes_)

# ----- features -----
X_df = df.drop(columns=[TARGET_COL]).copy()

# keep only numeric columns, remove non-numeric columns entirely
num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
removed_cat_cols = [c for c in X_df.columns if c not in num_cols]

if len(num_cols) == 0:
    raise ValueError("No numeric feature columns found after removing target and non-numeric columns.")

X_num = X_df[num_cols].apply(pd.to_numeric, errors="coerce")
X_num = X_num.fillna(X_num.median(numeric_only=True))
X_num = X_num.apply(pd.to_numeric, errors="coerce").fillna(0.0)

scaler = StandardScaler()
X_all = scaler.fit_transform(X_num.values).astype(np.float32, copy=False)
X_all = np.asarray(X_all, dtype=np.float32, order="C")
y_all = np.asarray(y_all, dtype=np.int64)

print(f"[DATA] Raw shape={df.shape}")
print(f"[DATA] Processed X={X_all.shape} | C={NUM_CLASSES} | map={class_map}")
print(f"[DATA] Numeric cols kept={len(num_cols)} | Removed non-numeric cols={len(removed_cat_cols)}")
if len(removed_cat_cols) > 0:
    print(f"[DATA] Removed columns: {removed_cat_cols}")
print(
    "[GPU ] cuda_available=", torch.cuda.is_available(),
    "| visible_gpus=", torch.cuda.device_count(),
    "| using_device=", DEVICE_STR
)

if NUM_CLASSES != 2:
    print(f"[WARN] Expected 2 classes, but found {NUM_CLASSES} classes.")

rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=SEED)

m_full = X_all.shape[1]


def build_ordering(params: dict):
    go_metric = params["go_metric"]
    go_k = int(params["go_num_clusters"])
    go_refine_passes = int(params["go_refine_passes"])
    go_direction = bool(params["go_direction_select"])
    go_feat_subsample = int(params["go_feat_subsample"])

    rng = np.random.default_rng(SEED + 999)
    if go_feat_subsample < m_full:
        feat_idx = rng.choice(m_full, size=go_feat_subsample, replace=False)
        feat_idx.sort()
    else:
        feat_idx = np.arange(m_full)

    X_go = X_all[:, feat_idx]

    go = GraphFeatureOrdering(
        num_clusters=go_k,
        metric=go_metric,
        refine=True,
        direction_select=go_direction,
        refine_passes=go_refine_passes,
    )

    try:
        Pi_sub, _, _, _ = go.fit(X_go, seed=SEED, deterministic=True, use_cpu_kmeans=False)
    except Exception:
        cleanup_cuda()
        Pi_sub, _, _, _ = go.fit(X_go, seed=SEED, deterministic=True, use_cpu_kmeans=True)

    ordered_subset = feat_idx[np.array(Pi_sub, dtype=np.int64)].tolist()

    if go_feat_subsample < m_full:
        remaining = np.setdiff1d(np.arange(m_full), feat_idx, assume_unique=False)
        Pi_star = ordered_subset + remaining.tolist()
    else:
        Pi_star = ordered_subset

    return Pi_star


def evaluate_with_params(params: dict, record_fold_metrics: bool = True):
    seed_everything(SEED)
    cleanup_cuda()

    build_start = time.perf_counter()
    Pi_star = build_ordering(params)
    order_time = time.perf_counter() - build_start

    tabpfn_seed = int(params["tabpfn_seed"])
    head_cfg = TabPFN25Config(
        task_type="binary",
        num_classes=int(NUM_CLASSES),
        device=TABPFN_DEVICE,
        random_state=tabpfn_seed,
    )

    accs = []
    f1s = []
    aucs = []

    t0 = time.perf_counter()

    for fold_id, (tr_idx, va_idx) in enumerate(rkf.split(X_all, y_all), start=1):
        X_tr = X_all[tr_idx]
        y_tr = y_all[tr_idx]
        X_va = X_all[va_idx]
        y_va = y_all[va_idx]

        nsc = PIDFSegPCA(
            segmentation=params["nsc_segmentation"],
            l_min=int(params["nsc_lmin"]),
            m_rule=params["nsc_m_rule"],
            gamma=float(params["nsc_gamma"]),
            beta=float(params["nsc_beta"]),
            tau=float(params["nsc_tau"]),
            M_min=int(params["nsc_Mmin"]),
            M_max=int(params["nsc_Mmax"]),
            assume_standardized=bool(params["assume_standardized"]),
            device=NSC_DEVICE,
        )

        deltas = None
        if params["nsc_segmentation"] != "uniform":
            deltas = compute_deltas_adjacent_corr_chunked(X_tr, Pi_star, chunk=2048)

        X_tr_t = torch.from_numpy(X_tr)
        X_va_t = torch.from_numpy(X_va)

        nsc.configure(
            Pi_star=Pi_star,
            X_train=X_tr_t,
            tau=float(params["nsc_tau"]),
            deltas=deltas
        )

        Z_tr = nsc.compress(X_tr_t, mode="flatten").cpu().numpy()
        Z_va = nsc.compress(X_va_t, mode="flatten").cpu().numpy()

        head = TabPFN25Head(head_cfg)
        head.fit(Z_tr, y_tr)

        P = head.predict_proba(Z_va)
        pred = np.argmax(P, axis=1)

        acc = float(accuracy_score(y_va, pred))
        f1m = float(f1_score(y_va, pred, average="macro"))
        aucm = safe_binary_auc(y_va, P)

        accs.append(acc)
        f1s.append(f1m)
        aucs.append(aucm)

        if record_fold_metrics:
            print(
                f"[Fold {fold_id:02d}] "
                f"ACC={acc:.6f} | "
                f"Macro-F1={f1m:.6f} | "
                f"ROC-AUC={aucm:.6f}"
            )

        cleanup_cuda()

    cv_elapsed_sec = time.perf_counter() - t0

    results = {
        "acc_mean": float(np.mean(accs)),
        "acc_std": float(np.std(accs, ddof=1)),
        "f1_mean": float(np.mean(f1s)),
        "f1_std": float(np.std(f1s, ddof=1)),
        "auc_mean": float(np.nanmean(aucs)),
        "auc_std": float(np.nanstd(aucs, ddof=1)),
        "order_time_sec": float(order_time),
        "cv_time_sec": float(cv_elapsed_sec),
        "elapsed_sec": float(order_time + cv_elapsed_sec),
        "fold_accs": accs,
        "fold_f1s": f1s,
        "fold_aucs": aucs,
    }
    return results


print("\n================ USING FIXED BEST HYPERPARAMETERS ================")
for k, v in BEST_PARAMS.items():
    print(f"{k}: {v}")

print("\n================ FINAL 5x5 CV WITH FIXED BEST PARAMS ================")
final_results = evaluate_with_params(BEST_PARAMS, record_fold_metrics=True)

print("\n================ FINAL SUMMARY ================")
print(f"Device used: {DEVICE_STR}")
print(f"GO-LR ordering runtime      : {final_results['order_time_sec']:.2f} seconds")
print(f"5x5 CV runtime              : {final_results['cv_time_sec']:.2f} seconds")
print(f"Total runtime               : {final_results['elapsed_sec']:.2f} seconds ({final_results['elapsed_sec']/60:.2f} min)")
print(f"Accuracy                    : {final_results['acc_mean']:.6f} ± {final_results['acc_std']:.6f}")
print(f"Macro-F1                    : {final_results['f1_mean']:.6f} ± {final_results['f1_std']:.6f}")
print(f"ROC-AUC                     : {final_results['auc_mean']:.6f} ± {final_results['auc_std']:.6f}")

[DATA] Raw shape=(1427, 4323)
[DATA] Processed X=(1427, 4322) | C=2 | map={'0': 0, '1': 1}
[DATA] Numeric cols kept=4322 | Removed non-numeric cols=0
[GPU ] cuda_available= True | visible_gpus= 8 | using_device= cuda:3

================ USING FIXED BEST HYPERPARAMETERS ================
go_metric: manhattan
go_num_clusters: 6
go_refine_passes: 2
go_direction_select: True
go_feat_subsample: 2000
nsc_segmentation: uniform
nsc_m_rule: default
nsc_tau: 0.95
nsc_gamma: 2.116958048807466
nsc_beta: 0.05755662243703091
nsc_Mmin: 16
nsc_Mmax: 384
nsc_lmin: 12
assume_standardized: True
tabpfn_seed: 1

================ FINAL 5x5 CV WITH FIXED BEST PARAMS ================
[Fold 01] ACC=0.888112 | Macro-F1=0.886867 | ROC-AUC=0.955375
[Fold 02] ACC=0.888112 | Macro-F1=0.886867 | ROC-AUC=0.946918
[Fold 03] ACC=0.880702 | Macro-F1=0.878425 | ROC-AUC=0.938705
[Fold 04] ACC=0.908772 | Macro-F1=0.908365 | ROC-AUC=0.963675
[Fold 05] ACC=0.891228 | Macro-F1=0.890315 | ROC-AUC=0.950099
[Fold 06] ACC=0.902098